[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rudrite/kernels/blob/main/labs/xla/lab-x2-tpu-pipeline.ipynb)

# LAB·X2 · The TPU pipeline, dumped

**Hardware:** Colab **TPU runtime** (Runtime -> Change runtime type -> TPU). LAB·X1 read the dump driver on CPU; this lab points the same driver at a real TPU and compares the two pipelines directly.

Passes are hardware-aware. A pipeline built for CPU is not the same list of names as a pipeline built for TPU, and the dump proves it either way. This lab also asks the driver to fail on purpose, once, so a real out-of-memory error lands in your hands instead of staying a warning in a slide.

Before you run anything: guess whether disabling fusion will make the compiled function faster or slower, and by roughly how much. Write the number down; the next section measures it for real.

In [ ]:
# Colab's preinstalled jax/libtpu pairing sometimes cannot read the pallas
# bytecode a fresh jax build emits. Installing a matched pair up front avoids
# a confusing failure later. If this installs or upgrades anything:
# Runtime -> Restart session, then Run all.
!pip install -q -U "jax[tpu]"
import importlib.metadata as md
print("jax", md.version("jax"))

**your prediction:**

One constraint before the code: `XLA_FLAGS` is read once, the first time a process touches a backend, and a single TPU chip only accepts one process at a time. Comparing "fusion on" against "fusion off" inside one already-running kernel would silently reuse whichever flags were set first. So both timings below run as their own fresh subprocess, before this notebook's own kernel has imported jax at all. Each subprocess starts clean, gets its answer, and exits, freeing the chip for the next one.

In [ ]:
import subprocess, sys, textwrap

TIMING_SCRIPT = textwrap.dedent("""
    import os
    os.environ["XLA_FLAGS"] = "{flags}"
    import time
    import jax
    import jax.numpy as jnp

    def attend(q, k, v):
        s = q @ k.T / jnp.sqrt(jnp.float32(q.shape[-1]))
        return jax.nn.softmax(s) @ v

    S, D = 2048, 128
    q = jax.random.normal(jax.random.key(0), (S, D), jnp.bfloat16)
    k = jax.random.normal(jax.random.key(1), (S, D), jnp.bfloat16)
    v = jax.random.normal(jax.random.key(2), (S, D), jnp.bfloat16)

    fn = jax.jit(attend)
    fn(q, k, v).block_until_ready()
    times = []
    for _ in range(20):
        t0 = time.perf_counter()
        fn(q, k, v).block_until_ready()
        times.append(time.perf_counter() - t0)
    print(sorted(times)[len(times) // 2] * 1e6)
""")

def timed_subprocess(flags):
    script = TIMING_SCRIPT.format(flags=flags)
    out = subprocess.run([sys.executable, "-c", script], capture_output=True, text=True, timeout=300)
    if out.returncode != 0:
        print(out.stderr[-2000:])
        raise RuntimeError("subprocess failed; see stderr above")
    return float(out.stdout.strip().splitlines()[-1])

COMPILE_SCRIPT = textwrap.dedent("""
import os
os.environ["XLA_FLAGS"] = "{flags}"
import jax
import jax.numpy as jnp
def attend(q, k, v):
    s = q @ k.T / jnp.sqrt(jnp.float32(q.shape[-1]))
    return jax.nn.softmax(s) @ v
x = jnp.ones((2048, 128), jnp.bfloat16)
print(jax.jit(attend).lower(x, x, x).compile().as_text())
""")

def compiled_text(flags):
    """The optimized module this flag setting produces, as text."""
    out = subprocess.run([sys.executable, "-c", COMPILE_SCRIPT.format(flags=flags)],
                         capture_output=True, text=True, timeout=600)
    if out.returncode != 0:
        print(out.stderr[-2000:])
        raise RuntimeError("compile subprocess failed; see stderr above")
    return out.stdout

FLAG = "--xla_disable_hlo_passes=fusion"

# Prove the flag moved something before timing anything. A pass name the
# backend does not register is accepted and ignored in silence, and a ratio
# measured across two identical modules says nothing about fusion.
on_text = compiled_text("")
off_text = compiled_text(FLAG)
changed = on_text != off_text
print("the flag changed the module:", changed)
if not changed:
    print()
    print("Both compilations are identical, so this backend has no pass under")
    print("that name and the flag did nothing. Any timing below compares the")
    print("program against itself. Find the real stage name in the pass list")
    print("this lab dumps further down, then disable that instead.")

fused_us = timed_subprocess("")
unfused_us = timed_subprocess(FLAG)
print()
print(f"pipeline untouched: {fused_us:.1f} us (median of 20)")
print(f"with the flag:      {unfused_us:.1f} us (median of 20)")
print(f"ratio: {unfused_us / fused_us:.2f}x" + ("" if changed else "   <- meaningless: same module both times"))

# Order control: if the gap follows the order rather than the flag, it is
# the harness talking, not the compiler.
first_again = timed_subprocess(FLAG)
second_again = timed_subprocess("")
print()
print(f"reversed order, with the flag: {first_again:.1f} us")
print(f"reversed order, untouched:     {second_again:.1f} us")

Chapter 5's claim is that fusion pays for itself by keeping intermediates out of HBM. So fusion off should come out slower here, with the broadcast, the divide, and the rest of the softmax chain each round-tripping through memory on their own.

LAB·X4 has since found the names that work on a v6e, `tpu_fusion` and `tpu_multi_output_fusion`, so try those rather than the flag below if you want a real comparison. Check the first line of the output before you read the numbers. On a v6e this flag turned out to change nothing at all: the same 66 instructions and the same seven fusions came back either way, because the TPU pipeline registers no pass under that name and XLA ignores the flag without comment. The timings that followed differed by 26% while compiling the identical program, which is a fact about the harness rather than about fusion.

That is why the cell diffs the modules first and why it repeats the pair in the opposite order. If your gap follows the order rather than the flag, the harness is talking. If the flag does move your module, then the comparison is real and the ratio means what it says. Either way, write down the chip and what you saw.

## the open question: what did each version dispatch?

The timing at the top of this lab came out backwards from what chapter 5 predicts, and the site publishes that disagreement rather than hiding it: on a v6e, this attention ran `187.5` microseconds with the pipeline untouched and `139.3` microseconds with the fusion pass disabled. Nobody has explained it yet, and this section is the first real attempt.

The hypothesis worth testing first: the backend may be pattern-matching this shape and dispatching its own kernel, the way the kernel path's profile caught XLA:TPU substituting an online-softmax custom call for exactly this program. If that substitution happens in one version and not the other, then the two runs are not the same computation with and without fusion, and the ratio is measuring a swap rather than a pass.

This section sits here, before the dump section below, for the same one-process reason as the timing: once this notebook's kernel imports jax and touches the chip, no fresh subprocess can compile on it, and both runs die with "The TPU is already in use". The cell below compiles both versions in fresh subprocesses (the same flag isolation the timing used) and prints what each one actually emitted: how many instructions, which custom calls by name, how many fusions, and the root instruction. Both modules are written to `/tmp` so you can diff them by hand afterwards. Whatever you find, paste it back: a confirmed answer closes the question in chapter 5, and a negative result narrows it.

In [ ]:
# Colab TPU runtime only
import subprocess, sys, textwrap

INSPECT = textwrap.dedent("""
import os, re, collections
os.environ["XLA_FLAGS"] = "{flags}"
import jax
import jax.numpy as jnp

def attend(q, k, v):
    s = q @ k.T / jnp.sqrt(jnp.float32(q.shape[-1]))
    return jax.nn.softmax(s) @ v

S, D = 2048, 128
x = jnp.ones((S, D), jnp.bfloat16)
text = jax.jit(attend).lower(x, x, x).compile().as_text()

open("{out}", "w").write(text)
print("instructions:", sum(1 for l in text.splitlines() if " = " in l))
print("fusions:", text.count(" fusion("))
targets = sorted(set(re.findall(r'custom_call_target="([^"]+)"', text)))
print("custom calls:", targets or "none")
# the entry computation's root, not a reduction region's: find ENTRY first
lines = text.splitlines()
start = next((i for i, l in enumerate(lines) if l.startswith("ENTRY")), 0)
root = next((l.strip() for l in lines[start:] if l.strip().startswith("ROOT")), "not found")
print("entry root:", root[:140])
""")

ok = True
for label, flags, out in [
    ("fusion on ", "", "/tmp/hlo-fusion-on.txt"),
    ("fusion off", "--xla_disable_hlo_passes=fusion", "/tmp/hlo-fusion-off.txt"),
]:
    print("=" * 60)
    print(label)
    print("=" * 60)
    r = subprocess.run([sys.executable, "-c", INSPECT.format(flags=flags, out=out)],
                       capture_output=True, text=True, timeout=600)
    print(r.stdout.strip() or r.stderr[-1500:])
    ok = ok and r.returncode == 0

print()
if ok:
    print("both modules saved: /tmp/hlo-fusion-on.txt and /tmp/hlo-fusion-off.txt")
else:
    print("a subprocess failed, so the modules were not written; fix the failure and rerun")

In [ ]:
import os
os.environ["XLA_FLAGS"] = "--xla_dump_to=/tmp/xla-dump-tpu --xla_dump_hlo_pass_re=.*"

import jax
import jax.numpy as jnp

print(jax.__version__, jax.devices())
assert jax.devices()[0].platform == "tpu", "Runtime -> Change runtime type -> TPU, then Run all"

def attend(q, k, v):
    s = q @ k.T / jnp.sqrt(jnp.float32(q.shape[-1]))
    return jax.nn.softmax(s) @ v

x = jnp.ones((64, 64))
jax.jit(attend).lower(x, x, x).compile()
print("compiled on TPU. /tmp/xla-dump-tpu now holds one file per pass step.")

**your prediction:**
Will this TPU dump hold more files than the 42 CPU produced in LAB·X1, or fewer? Which fact about codegen (chapter 9) makes you expect that direction?

In [ ]:
# captured once on CPU, jax 0.4.38, the same attend program as LAB·X1;
# your own CPU run may carry a different module_ prefix, but these are the
# per-step suffixes LAB·X1 found for the numbered pass files
CPU_REFERENCE_STEPS = [
    "0000.sharding-removal.after_pipeline-start.before_sharding-remover",
    "0001.SubbytePacker_pipeline.after_pipeline-start.before_sub-byte-size-setter",
    "0002.HLO_passes_through_layout_assignment.after_pipeline-start.before_gather_scatter_normalizer",
    "0003.simplification.after_pipeline-start.before_algsimp",
    "0004.simplification.after_algsimp.before_simplify-sorts",
    "0005.simplification.after_tree_reduction_rewriter.before_zero_sized_hlo_elimination",
    "0006.simplification.after_pipeline-start.before_algsimp",
    "0007.simplification.after_algsimp.before_simplify-sorts",
    "0008.simplification.after_pipeline-start.before_algsimp",
    "0009.HLO_passes_through_layout_assignment.after_simplification.before_bitcast_dtypes_expander",
    "0010.HLO_passes_through_layout_assignment.after_transpose-folding.before_cse",
    "0011.HLO_passes_through_layout_assignment.after_cse.before_cse_barrier_expander",
    "0012.HLO_passes_through_layout_assignment.after_flatten-call-graph.before_layout-assignment",
    "0013.HLO_passes_through_layout_assignment.after_layout-assignment.before_sub-byte-size-setter",
    "0014.hlo_normalization.after_pipeline-start.before_reshape-decomposer",
    "0015.HLO_passes_after_layout_assignment.after_pipeline-start.before_after_layout_assignment",
    "0016.after_layout_assignment.after_pipeline-start.before_pipeline-end",
    "0017.HLO_passes_after_layout_assignment.after_fusion.before_simplification_after_layout_assignment",
    "0018.simplification_after_layout_assignment.after_pipeline-start.before_algsimp",
    "0019.copy-insertion.after_adding_copies_to_resolve_interference",
    "0020.copy-insertion.after_removing_unnecessary_copies",
    "0021.copy-insertion.after_adding_special-case_copies",
    "0022.HLO_passes_after_layout_assignment.after_copy-insertion.before_dce",
]

import re

DUMP_DIR = "/tmp/xla-dump-tpu"
all_files = sorted(os.listdir(DUMP_DIR))
attend_files = sorted(f for f in all_files if "jit_attend" in f)
print(f"{len(attend_files)} files belong to attend on this TPU run (CPU produced 42)")

print()
print("what the filenames actually look like here:")
for f in attend_files[:5]:
    print("   ", f)
print()

# TPU files carry a build id between the module name and the step number:
#   module_0007.jit_attend.cl_948136882.0000.hlo_device_type_async_wrapper...
# CPU files have no such segment. Drop it so both backends produce step
# names that can actually be compared.
pattern = re.compile(r"jit_attend\.(?:cl_\d+\.)?(\d{4}\..+)\.txt$")
tpu_steps = sorted({m.group(1) for f in attend_files if (m := pattern.search(f))})
cpu_steps = set(CPU_REFERENCE_STEPS)

if not tpu_steps:
    print("no step names parsed: this backend names its dump files another way.")
    print("Read the sample above and adjust the pattern before trusting any diff.")

tpu_only = sorted(set(tpu_steps) - cpu_steps)
cpu_only = sorted(cpu_steps - set(tpu_steps))
print(f"{len(tpu_steps)} step name(s) parsed on this run")
print(f"{len(tpu_only)} appear here but not in the CPU reference")
for s in tpu_only[:10]:
    print("  +", s)
print(f"{len(cpu_only)} appear in the CPU reference but not here")
for s in cpu_only[:10]:
    print("  -", s)

A first run of this cell on a v6e parsed zero step names, which is why it prints the filenames before it trusts anything. The reason turned out to be a segment CPU dumps do not have: TPU files carry a build id, as in `module_0007.jit_attend.cl_948136882.0000.hlo_device_type_async_wrapper...`, and the pattern above now drops it so the two backends produce comparable names. Check the sample against your own run before believing the diff, because a different libtpu build may file things differently again. CPU emits its own LLVM IR into a thunk runtime; that machinery, and the passes that prepare for it, has no reason to exist on the TPU side of the pipeline. TPU instead hands its fusions to libtpu once optimization finishes, and libtpu is closed: none of what happens after that hand-off shows up in an HLO-level dump, on either backend. The exact step names you see depend on your assigned TPU generation and its libtpu build, so use the reference list as a diffing tool, not a fixed answer key: the fact worth writing down is which named stages the two backends share and which they do not, not matching an exact count.

## capture for the museum

Every lab so far has asked you to predict a real number and check it. This last one asks you to break something on purpose. Allocate an array sized comfortably past this chip's HBM, inside a jitted function, and XLA has no choice but to refuse: `RESOURCE_EXHAUSTED`, with the exact byte counts it tried and failed to place.

That error text is not noise to swallow. It is the same kind of artifact as the dump files above, a real compiler decision, this time a refusal, with full provenance. Run the next two cells, then copy the verbatim error text and the failing snippet straight into the paste-back block at the end. Captures like this become exhibits in the site's museum, in the XLA wing: real failures, kept whole, instead of a paraphrase of what the error probably said.

In [ ]:
BIG = 300_000  # BIG*BIG*4 bytes ~ 360 GB, comfortably past any single TPU chip's HBM

@jax.jit
def blow_up(x):
    return x + 1.0

museum_capture = {}
try:
    huge = jnp.ones((BIG, BIG), dtype=jnp.float32)
    out = blow_up(huge)
    out.block_until_ready()
    museum_capture["status"] = "did not fail; raise BIG and rerun this cell"
    print(museum_capture["status"])
except Exception as e:
    museum_capture["status"] = "failed as expected"
    museum_capture["error_text"] = str(e)
    museum_capture["snippet"] = (
        f"BIG = {BIG}\n"
        "huge = jnp.ones((BIG, BIG), dtype=jnp.float32)\n"
        "out = jax.jit(lambda x: x + 1.0)(huge)\n"
    )
    print(museum_capture["error_text"][:2000])

In [ ]:
print("=" * 60)
print("MUSEUM EXHIBIT: paste this whole blob into the site's museum XLA wing")
print("=" * 60)
print("chip:", jax.devices()[0].device_kind)
print()
print("failing snippet:")
print(museum_capture.get("snippet", "(cell above did not fail; raise BIG and rerun it first)"))
print()
print("verbatim error text:")
print(museum_capture.get("error_text", "(none captured yet)"))

## mark it run

Chapter 05 (kernels.rudrite.com/xla/fusion) is the timing you just measured, argued in prose against three more before-and-after pairs pulled from this same TPU family. Chapter 09 (kernels.rudrite.com/xla/codegen) is the hand-off you found between CPU thunks and libtpu. Your museum capture belongs on the site the moment it exists: paste it in, and LAB·X3 goes looking for the collectives that same partitioner inserts.